In [0]:
# Prediction Pipeline - Automated via CI/CD
import pickle
from datetime import datetime, timedelta
import sys

# Add project root to path
sys.path.insert(0, '/Workspace/Repos/rathodshyam2301@gmail.com/MLProject_Databricks')

from src.utils.config import load_config
from src.models.predict import predict

In [0]:
config = load_config()
print(f"Train table: {config['data']['train_table']}")
print(f"Predict table: {config['data']['predict_table']}")
print(f"Output table: {config['data']['output_table']}")

In [0]:
model_row = spark.table(config["model"]["model_table"]).select("model_blob").first()
if model_row is None:
    raise RuntimeError("No trained model found. Run training first.")

model = pickle.loads(model_row["model_blob"])

print("=" * 60)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Model Type: {type(model).__name__}")
print(f"Model Coefficients: {model.coef_}")
print(f"Model Intercept: {model.intercept_}")
print("=" * 60)

In [0]:
# Generate predictions
predictions_df = predict(
    data_table=config["data"]["predict_table"],
    spark=spark,
    model=model
)

# Add IST timestamp column (UTC + 5:30)
ist_timestamp = datetime.utcnow() + timedelta(hours=5, minutes=30)
predictions_df['prediction_timestamp'] = ist_timestamp

# Print predictions
print("\n" + "=" * 60)
print("PREDICTIONS GENERATED")
print("=" * 60)
print(f"Number of predictions: {len(predictions_df)}")
print(f"Timestamp (IST): {ist_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
print("\nFirst 5 predictions:")
print(predictions_df.head().to_string())
print("=" * 60)

In [0]:
# Save predictions to table with schema overwrite
spark.createDataFrame(predictions_df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    config["data"]["output_table"]
)

print(f"\n✓ Predictions saved successfully to: {config['data']['output_table']}")
print(f"✓ Timestamp (IST): {ist_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Total predictions: {len(predictions_df)}")